In [35]:
import tensorflow as tf
import pandas as pd
import numpy as np
import sys, os
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD

In [36]:
mnist = tf.keras.datasets.mnist
(xtrain, ytrain), (xtest, ytest) = mnist.load_data()

xtrain, xtest = xtrain/255.0 * 2 -1, xtest/255.0 * 2 - 1

In [37]:
xtrain.shape

(60000, 28, 28)

In [38]:
N, H, W = xtrain.shape
D = H*W
xtrain = xtrain.reshape(-1, D)
xtest = xtrain.reshape(-1, D)

In [39]:
xtrain.shape

(60000, 784)

In [40]:
latent_dim = 100

In [41]:
def build_generator(latent_dim):
  i = Input(shape=(latent_dim,))
  x = Dense(256, activation=LeakyReLU(negative_slope=0.2))(i)
  x = BatchNormalization(momentum=0.8)(x)
  x = Dense(512, activation=LeakyReLU(negative_slope=0.2))(x)
  x = BatchNormalization(momentum=0.8)(x)
  x = Dense(1024, activation=LeakyReLU(negative_slope=0.2))(x)
  x = BatchNormalization(momentum=0.8)(x)
  x = Dense(D, activation='tanh')(x)

  model = Model(i, x)
  return model

In [42]:
def build_discriminator(img_size):
  i = Input(shape=(img_size,))
  x = Dense(512, activation=LeakyReLU(negative_slope=0.2))(i)
  x = Dense(256, activation=LeakyReLU(negative_slope=0.2))(x)
  x = Dense(1, activation='sigmoid')(x)

  model = Model(i, x)
  return model

In [43]:
discriminator = build_discriminator(D)
discriminator.compile(loss='binary_crossentropy',
                      optimizer=SGD(0.0002, 0.5),
                      metrics=['accuracy'])

generator = build_generator(latent_dim)

z = Input(shape=(latent_dim,))
img = generator(z)

fake_pred = discriminator(img)

combined_model = Model(z, fake_pred)
combined_model.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))

In [44]:
batch_size = 32
epochs = 50000
sample_period = 200

ones = np.ones(batch_size)
zeros = np.zeros(batch_size)

d_losses = []
g_losses = []

if not os.path.exists('gan_images'):
  os.makedirs('gan_images')



In [45]:
# this is to just look back at when the model was training
def sample_images(epoch):
  rows, cols = 5, 5
  noise = np.random.randn(rows*cols, latent_dim)
  imgs = generator.predict(noise, verbose=0)

  imgs = 0.5 * imgs + 0.5

  fig, axs = plt.subplots(rows, cols)
  idx = 0
  for i in range(rows):
    for j in range(cols):
      axs[i, j].imshow(imgs[idx].reshape(H, W), cmap='gray')
      axs[i, j].axis('off')
      idx += 1
  fig.savefig("gan_images/%d.png" % epoch)
  plt.close()

In [ ]:
#main training loops:

for epoch in range(epochs):

  # train dicriminator

  discriminator.trainable = True

  idx = np.random.randint(0, xtrain.shape[0], batch_size)
  real_imgs = xtrain[idx]

  noise = np.random.randn(batch_size, latent_dim)
  fake_imgs = generator.predict(noise, verbose=0)

  d_loss_real, d_acc_real = discriminator.train_on_batch(real_imgs, ones)
  d_loss_fake, d_acc_fake = discriminator.train_on_batch(fake_imgs, zeros)
  d_loss = 0.5 * (d_loss_real + d_loss_fake)
  d_acc = 0.5 * (d_acc_real + d_acc_fake)

  # train generator

  discriminator.trainable = False

  noise = np.random.randn(batch_size, latent_dim)
  g_loss1 = combined_model.train_on_batch(noise, ones)

  noise = np.random.randn(batch_size, latent_dim)
  g_loss2 = combined_model.train_on_batch(noise, ones)

  g_loss = 0.5 * (g_loss1 + g_loss2)

  d_losses.append(d_loss)
  g_losses.append(g_loss)

  if epoch % 100 == 0:
    print(f"epoch: {epoch+1}/{epochs}, d_loss: {d_loss:.2f}, d_acc: {d_acc:.2f}, g_loss: {g_loss:.2f}")

  if epoch % sample_period == 0:
    sample_images(epoch)

epoch: 1/50000, d_loss: 0.40, d_acc: 0.75, g_loss: 0.44
epoch: 101/50000, d_loss: 1.55, d_acc: 0.50, g_loss: 0.08
epoch: 201/50000, d_loss: 1.62, d_acc: 0.50, g_loss: 0.06
epoch: 301/50000, d_loss: 1.55, d_acc: 0.49, g_loss: 0.07
epoch: 401/50000, d_loss: 1.49, d_acc: 0.49, g_loss: 0.09
epoch: 501/50000, d_loss: 1.43, d_acc: 0.47, g_loss: 0.11
epoch: 601/50000, d_loss: 1.37, d_acc: 0.46, g_loss: 0.13
epoch: 701/50000, d_loss: 1.31, d_acc: 0.46, g_loss: 0.15
epoch: 801/50000, d_loss: 1.26, d_acc: 0.46, g_loss: 0.18
epoch: 901/50000, d_loss: 1.22, d_acc: 0.46, g_loss: 0.20
epoch: 1001/50000, d_loss: 1.18, d_acc: 0.46, g_loss: 0.22
epoch: 1101/50000, d_loss: 1.15, d_acc: 0.46, g_loss: 0.24
epoch: 1201/50000, d_loss: 1.12, d_acc: 0.46, g_loss: 0.25
epoch: 1301/50000, d_loss: 1.09, d_acc: 0.46, g_loss: 0.27
epoch: 1401/50000, d_loss: 1.07, d_acc: 0.46, g_loss: 0.29
epoch: 1501/50000, d_loss: 1.05, d_acc: 0.46, g_loss: 0.30
epoch: 1601/50000, d_loss: 1.03, d_acc: 0.46, g_loss: 0.31
epoch: 17